# 6.2 Appearance descriptors

In the previous notebook we described regions by their shape, using only their binary mask or their boundary. **Appearance descriptors**, instead, characterize the **pixel values inside the region**: which intensities or colors are present (**intensity/color statistics**) and how they are spatially arranged (**texture**), as shown in Fig. 1. Notice that, unlike shape descriptors, they don't require binary images as input.

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/region_description/images/appearance_overview.png" width="750"/>
    <figcaption>Fig. 1. The appearance of a region can be described by its intensity/color (pixel values, histogram, color statistics) and by its texture (spatial arrangement and repeated local patterns).</figcaption>
</center>$\\[5pt]$

This notebook covers different appearance description techniques:

- Intensity statistics and histogram moments (<a href="#621">section 6.2.1</a>)
- Texture description with the Gray-Level Co-occurrence Matrix (GLCM) (<a href="#622">section 6.2.2</a>)
- The invariance of both descriptors to rotation and scale (<a href="#623">section 6.2.3</a>)

## Problem context - Identification of the state of a license plate

An American company contacted us to develop a **texture description method** that describes **a license plate according to its state of origin**, instead of the characters on it. As you may know, the USA uses a different license plate for each state in the country (see Fig. 2).

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/region_description/images/usa_plates.jpg" width="600"/>
    <figcaption>Fig. 2. License plates from different states of the USA.</figcaption>
</center>$\\[5pt]$

You will apply some appearance description methods to this problem, like **histogram moments** or **co-occurrence matrices**. Your task is to develop a method returning a feature vector $\mathbf{f}$ that allows for the identification of the state of origin of such license plates.

> 🔍 <font color="#0072B2"><b>AI Exploration:</b> Texture description in practice</font>  
> **Suggested prompt**  
> *In the context of computer vision, in which applications is texture description used today (e.g. medical imaging, remote sensing or industrial inspection), and which texture descriptors are common besides the gray-level co-occurrence matrix?*

In [ ]:
# In case of using Google Colab, mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    # If your notebook and images are in your Drive, change directory accordingly:
    # %cd '/content/drive/MyDrive/Colab Notebooks/uma_cv_2027/Chapter 06. Region Description'
except ImportError:
    pass

import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (15.0, 8.0)

images_path = './images/'

## 6.2.1 Intensity statistics <a id="621"></a>

Intensity (and color) statistics describe **which pixel values are present in the region**. For a grayscale region $R$ with $N$ pixels of intensities $I(x,y)$, the most common ones are:

- **Mean intensity** (brightness):

  $$\mu_I = \frac{1}{N} \sum_{(x,y) \in R} I(x,y)$$

- **Variance** (contrast):

  $$\sigma_I^2 = \frac{1}{N} \sum_{(x,y) \in R} \left(I(x,y) - \mu_I\right)^2$$

- **Minimum and maximum intensity** (range): $I_{min}$ and $I_{max}$, the darkest and the brightest values of the region.

- **Percentiles**: the intensity below which a given percentage of the pixels of the region falls, e.g. the median is the 50th percentile.

- **Histogram** (distribution of values): $h(z)$, the number of pixels of the region with intensity $z$.

For color regions, the same statistics are computed per channel, e.g. the mean color $\boldsymbol{\mu}_C = [\mu_R, \mu_G, \mu_B]^\top$, along with the color covariance $\Sigma_C$ or color histograms, also in other color spaces (HSV, Lab, ...).

### Histogram moments

The **central moments of the histogram** describe the distribution of the intensities of the region compactly, with a few features. Dividing the histogram by the number of pixels, $p(z) = h(z)/N$, the central moment of order $n$ is:

$$\mu_n = \sum_{z=0}^{255} (z - \mu_I)^n \, p(z)$$

where the mean intensity can also be computed from the histogram, $\mu_I = \sum_{z=0}^{255} z \, p(z)$. Keep in mind that:

- $\mu_0 = 1$ and $\mu_1 = 0$,
- $\mu_2 = \sigma_I^2$ is the variance (contrast),
- $\mu_3$ measures the asymmetry (skewness) of the histogram, and
- $\mu_4$ measures how peaked or flat (kurtosis) the histogram is.

> 🛠️ <font color="#0B6B3A"><b>AI Coding Support:</b> Plotting histograms</font>  
> Remember that in Chapter 2 we already computed and plotted histograms with Matplotlib's [`plt.hist()`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.hist.html), so let's recall how it works!

> **Suggested prompt**  
> *I need to plot the histogram of a grayscale image (a 2D NumPy array of type uint8) with Matplotlib's `plt.hist()`. Do not write the full solution. Explain why the image has to be flattened first (e.g. with `np.ravel()`), and what the `bins` and `range` parameters mean when the intensities go from 0 to 255.*

### **<span style="color:green"><b><i>ASSIGNMENT 1: Analyzing histograms</i></b></span>**

In order to play a bit with these moments, let's go back to the state recognition of USA license plates. Two examples of such license plates are shown in Fig. 3.

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/region_description/images/nevada.jpg" width="300"/>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/region_description/images/hawaii.jpg" width="300"/>
    <figcaption>Fig. 3. License plates from Nevada (left) and Hawaii (right).</figcaption>
</center>$\[5pt]$

As we can see, the main difference between them is the texture in the plate background, as each state has a different one. Let's try the histogram moments for describing those textures!

**Your first task** is to plot the histograms of these images, `nevada.jpg` and `hawaii.jpg` (read in grayscale), and check if the shape of the histograms is enough to differentiate them.

In [ ]:
# Assignment 1

# Read images
nevada = cv2.imread(images_path + 'nevada.jpg', cv2.IMREAD_GRAYSCALE)
hawaii = cv2.imread(images_path + 'hawaii.jpg', cv2.IMREAD_GRAYSCALE)

# Show the first histogram
plt.subplot(121)
plt.title("Nevada plate")
plt.hist(None, bins=None, range=[None,None])

# And the second one!
plt.subplot(122)
plt.title("Hawaii plate")
plt.hist(None, bins=None, range=[None,None])
plt.show()

Now, let's complete the function `histogram_moments()`, which computes the central moments $\mu_n$ of the histogram of an image. It takes as input:

- an image, and
- the number of moments `k` to be computed ($\mu_0, \dots, \mu_{k-1}$),

and returns an array containing those moments of the image's histogram.

In [ ]:
def histogram_moments(image,k):
    """ Compute the central moments of the histogram of an image.

        Args:
            image: input image
            k: number of moments to compute (mu_0, ..., mu_{k-1})

        Returns:
            histogram_moments: array containing the histogram moments
    """

    # Compute the histogram h(z) and normalize it, p(z) = h(z)/N
    hist = cv2.calcHist([image],[0],None,[256],[0,256]).flatten().astype(np.float64) # Keep the None in this function!
    p = hist/hist.sum()

    # Compute the mean intensity (brightness) of the image
    mu_I = np.sum(np.arange(256)*p)

    # Compute moments
    histogram_moments = np.zeros(k)
    for n in range(k):
        moment = 0.0
        for z in range(256):
            moment += (None-None)**None * None

        histogram_moments[n] = moment

    return histogram_moments

You can use the next code to **test if the results are correct**:

In [ ]:
image = np.array([[10,60,20],[60,22,74],[72,132,2]], dtype=np.uint8)

moments = histogram_moments(image,6)

print(moments)

<font color='blue'>**Expected output:**  </font>

```
[ 1.00000000e+00 -1.77635684e-15  1.50795062e+03  3.83609108e+04
  6.08670794e+06  3.62329032e+08]
```

## 6.2.2 Texture: Gray-Level Co-occurrence Matrix (GLCM) <a id="622"></a>

Histograms describe **which values are present, but not where they occur**, so very different textures may have the same histogram, and therefore the same histogram moments (see Fig. 4).

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/region_description/images/same_histogram_textures.png" width="600"/>
    <figcaption>Fig. 4. Three patterns with very different textures but the same histogram.</figcaption>
</center>$\\[5pt]$

**Texture** describes the **spatial arrangement and repetition of the intensity/color values** of a region, providing information about its smoothness, coarseness and regularity (see Fig. 5). In this way, if a region does not present changes in intensity, we say that it is an untextured region.

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/region_description/images/examples_of_different_textures.png" width="700"/>
    <figcaption>Fig. 5. Examples of different textures.</figcaption>
</center>$\\[5pt]$

Unlike a histogram, the **Gray-Level Co-occurrence Matrix (GLCM)** $A(i,j)$ contains information about the spatial arrangement:

- $i$ and $j$ are **quantized gray levels**: the 256 original levels are often reduced to $L$ = 8, 16 or 32 levels, so $\dim(A) = L \times L$.
- $A(i,j)$ counts how often a pixel of gray level $i$ has a neighboring pixel of gray level $j$ at the **spatial offset** $\mathbf{d} = (\Delta x, \Delta y)$.

Fig. 6 shows an example with $L = 3$ gray levels and $\mathbf{d} = (1,1)$, that is, the neighbor is the pixel one position below and to the right (remember that the $y$ axis points down in images).

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/region_description/images/glcm_example.png" width="650"/>
    <figcaption>Fig. 6. GLCM of an image with $L = 3$ gray levels for the offset $\mathbf{d} = (1,1)$.</figcaption>
</center>$\\[5pt]$

To compute texture descriptors, the GLCM is first normalized, so its entries become the **probabilities of intensity co-occurrence**:

$$C(i,j) = \frac{A(i,j)}{\sum_{i,j} A(i,j)}$$

The denominator is the total number of pairs of pixels counted in the GLCM. For example, in Fig. 6 only the $4 \times 4 = 16$ pixels in the first 4 rows and columns have a neighbor at $\mathbf{d} = (1,1)$ inside the $5 \times 5$ image, so $\sum_{i,j} A(i,j) = 16$ and $C(i,j) = \frac{A(i,j)}{16}$.

Some statistics computed from the normalized GLCM that measure texture are (see Fig. 7):

- **Contrast**, high for strong local intensity differences:

  $$\text{Contrast} = \sum_{i,j} (i-j)^2 \, C(i,j)$$

- **Energy**, high for uniform or very regular textures:

  $$\text{Energy} = \sum_{i,j} C(i,j)^2$$

- **Homogeneity**, high when neighboring pixels have similar intensities:

  $$\text{Homogeneity} = \sum_{i,j} \frac{C(i,j)}{1+|i-j|}$$

- **Entropy**, high for complex or random textures (with $0 \log 0 = 0$):

  $$\text{Entropy} = -\sum_{i,j} C(i,j) \log C(i,j)$$

<center>
    <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/region_description/images/glcm_features.png" width="750"/>
    <figcaption>Fig. 7. GLCM-based texture descriptors, each with an example of a texture for which it is high.</figcaption>
</center>$\\[5pt]$

Together, they form the texture descriptor $\mathbf{f} = [f_1, f_2, f_3, f_4] = [\text{contrast}, \text{energy}, \text{homogeneity}, \text{entropy}]$.

**Practical issue:** the resulting GLCM descriptor depends on the chosen spatial offset $\mathbf{d}$ and on the gray-level quantization $L$ used to build the matrix.

> 🛠️ <font color="#0B6B3A"><b>AI Coding Support:</b> Building a GLCM with NumPy</font>  
> **Suggested prompt**  
> *I need to build the gray-level co-occurrence matrix of a grayscale image in Python for a spatial offset d = (dx, dy) whose components may be negative. Do not write the full solution. Explain how to choose the ranges of rows and columns so that both the pixel and its neighbor at offset d fall inside the image, and in which order NumPy indexes the rows and columns of an image.*

### **<span style="color:green"><b><i>ASSIGNMENT 2: Computing the GLCM</i></b></span>**

Let's implement the GLCM descriptor in two functions:

- `glcm()`, which takes an image whose gray levels are already quantized to $\{0, \dots, L-1\}$, the spatial offset `d` $= (\Delta x, \Delta y)$ (whose components may take positive or negative values) and the number of gray levels `L`, and returns the GLCM $A$ (an $L \times L$ matrix of counts).
- `glcm_features()`, which normalizes the GLCM $A$ to obtain $C$, and returns the texture descriptor $\mathbf{f}$ = `[contrast, energy, homogeneity, entropy]`.

Remember that NumPy indexes images as `image[y, x]`, that is, row first and then column. To quantize an image with 256 gray levels to $L$ levels, you can use the `quantize()` function below.

In [ ]:
def quantize(image, L):
    """ Quantize the 256 gray levels of an image to L levels (0, ..., L-1). """
    return (image.astype(np.uint16) * L // 256).astype(np.uint8)

In [ ]:
# Assignment 2
def glcm(image, d, L):
    """ Compute the Gray-Level Co-occurrence Matrix (GLCM) of an image.

        Args:
            image: grayscale image with gray levels in {0, ..., L-1}
            d: spatial offset (dx, dy)
            L: number of gray levels

        Returns:
            A: GLCM, L x L matrix of counts
    """

    dx, dy = d
    rows, cols = image.shape
    A = np.zeros((L,L))

    # Ranges of rows (y) and columns (x) where both (x,y) and (x+dx,y+dy) are inside the image
    range_y = range(max(0,-dy), min(rows,rows-dy))
    range_x = range(max(0,-dx), min(cols,cols-dx))

    # Count co-occurrences
    for y in range_y:
        for x in range_x:
            i = None
            j = None
            A[None,None] += 1

    return A


def glcm_features(A):
    """ Compute texture features from a GLCM.

        Args:
            A: GLCM, L x L matrix of counts

        Returns:
            f: texture descriptor [contrast, energy, homogeneity, entropy]
    """

    # Normalize the GLCM
    C = A/np.sum(A)

    # Gray levels i (row) and j (column) of each entry of C
    i, j = np.indices(C.shape)

    # Contrast
    contrast = None

    # Energy
    energy = None

    # Homogeneity
    homogeneity = None

    # Entropy (only over the non-zero entries, since 0 log 0 = 0)
    nonzero = C > 0
    entropy = None

    return np.array([contrast, energy, homogeneity, entropy])

You can use the next code, with the image and offset of Fig. 6, to **test if the results are right**:

In [ ]:
np.set_printoptions(suppress=True)

# Image and offset of Fig. 6
image = np.array([[0,0,1,2,2],
                  [1,0,1,1,2],
                  [2,2,1,0,0],
                  [1,1,0,2,0],
                  [0,0,2,0,1]], dtype=np.uint8)

A = glcm(image, d=(1,1), L=3)
print('A =\n', A)
print('f =', np.round(glcm_features(A),4))

<font color='blue'>**Expected output:**  </font>

```
A =
 [[3. 2. 0.]
 [3. 1. 3.]
 [1. 2. 1.]]
f = [0.875  0.1484 0.6458 1.9813]
```

### <font color="blue"><b><i>Thinking about it (1)</i></b></font>

Now, **answer the following question:**

- How would the descriptor change if you used another spatial offset $\mathbf{d}$ or another number of gray levels $L$?

    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

## 6.2.3 Invariance analysis <a id="623"></a>

Now that we have two appearance descriptors, the histogram moments and the GLCM features, let's check if they are invariant to rotation and scale. As in the number-plate problem of the previous notebook, the plates may appear at different sizes and orientations in the images, so our descriptor should be (at least) scale invariant.

### **<span style="color:green"><b><i>ASSIGNMENT 3: Checking the invariance of both descriptors</i></b></span>**

**What to do?** Build a rotated version (90 degrees) and a scaled version (by a factor of 2 in both axes) of the image `hawaii.jpg`. The code then computes, for each version, the first six central moments of its histogram and its GLCM features (with $L = 8$ gray levels and the offset $\mathbf{d} = (1,1)$), so you can compare them.

*Hint: You can rotate a NumPy array using [`np.rot90()`](https://numpy.org/doc/stable/reference/generated/numpy.rot90.html) and scale an image using [`cv2.resize()`](https://docs.opencv.org/4.x/da/d54/group__imgproc__transform.html#ga47a974309e9102f5f08231edc7e7529d).*

In [ ]:
# Assignment 3
np.set_printoptions(precision=4, suppress=False)
L = 8
d = (1,1)

# Read the image and build its transformed versions
image = cv2.imread(images_path + 'hawaii.jpg', cv2.IMREAD_GRAYSCALE)
versions = {
    'Original': image,
    'Rotated': np.rot90(None),
    'Scaled': cv2.resize(None, dsize=None, fx=None, fy=None, interpolation = cv2.INTER_AREA), # keep the dsize=None and the interpolation
}

# Compute and compare both descriptors
for name, version in versions.items():
    moments = histogram_moments(None, 6)
    features = glcm_features(glcm(quantize(None, L), d, L))
    print(name)
    print('  Histogram moments:', moments)
    print('  GLCM features:    ', np.round(features, 4))

### <font color="blue"><b><i>Thinking about it (2)</i></b></font>

Now, **answer the following questions:**

- Which descriptor is invariant to rotation? And to scale? **Explain why**, thinking about the information each one uses.

    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

- Which descriptor would you use to identify the state of a plate? **Why?**

    <p style="margin: 4px 0px 0px 5px; color:blue"><i>Your answer here!</i></p>

> 🧭 <font color="#B8860B"><b>AI Reflection:</b> Histogram moments vs GLCM</font>  
> **Suggested prompt**  
> *I have described license plates with the central moments of their histogram and with GLCM features (contrast, energy, homogeneity and entropy), and checked their invariance to rotation and scale. Guide me, without giving me the answers, to reason why the GLCM features change with rotation and scale while the histogram moments do not, and which descriptor would better identify the state of a plate.*

✍️ **Your notes / key takeaways:**

<p style="margin: 4px 0px 6px 5px; color:blue"><i>Write your conclusions here after discussing with your assistant!</i></p>

### **<span style="color:green"><b><i>OPTIONAL</i></b></span>**

Usually, the GLCM is computed for several offsets with the same length and different directions, e.g. $\mathbf{d} = (1,0)$, $(1,1)$, $(0,1)$ and $(-1,1)$ (0, 45, 90 and 135 degrees), and the features are averaged over them. **What do you think this is due to?**

**Implement this new procedure** and then check the invariances again. **What happened?**

### **<span style="color:green"><b><i>END OF OPTIONAL PART</i></b></span>**

## Conclusion

Awesome! This was a laborious and dense notebook, but you carried it through to the end!

In this notebook you have learned:

- how to describe the appearance of a region with intensity statistics and the central moments of its histogram,
- how to describe its texture with the GLCM and the contrast, energy, homogeneity and entropy computed from it, and
- how invariant both descriptors are to rotation and scale, in the context of the state identification problem.

## <font color="orange"><b><i>AI Appendix</i></b></font>

- **Model/tool used:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

- **What I used AI for:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Briefly explain how you used AI in this notebook: whether you followed the proposed AI interactions, adapted them, skipped some of them, or used AI for additional purposes such as debugging, clarifying concepts, improving prompts or checking your reasoning.</i></p>

- **Most and least useful proposed AI interactions:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>By proposed AI interactions, we mean the AI prompts explicitly included in the notebook. Choose one that helped you the most and one that helped you the least. Briefly explain why.</i></p>

- **Most useful personal AI interaction:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>By personal AI interaction, we mean any additional prompt or follow-up question that you wrote yourself, beyond the prompts proposed in the notebook. Paste or summarize one, if any, and explain why it helped.</i></p>

- **What I verified, adapted or decided myself:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Briefly explain what you checked, corrected, adapted or decided before using the AI output in your final work.</i></p>